# ⚖️ Adversa — GRPO Training Notebook

**Multi-Agent Adversarial Courtroom Environment**  
OpenEnv Hackathon India Finale 2026

---

## 🔧 Setup
1. Set `ENV_URL` to your HF Space URL
2. Run all cells top-to-bottom
3. Training takes ~4-6 hours on A100

In [ ]:
# @title ⚙️ Cell 1 — Install dependencies
!pip install -q unsloth trl transformers requests matplotlib datasets

In [ ]:
# @title 🔗 Cell 2 — Configuration
import requests, json, re, random, time
import matplotlib.pyplot as plt
import numpy as np

# ⚠️ CHANGE THIS to your deployed HF Space URL
ENV_URL = 'https://dorare22-adversa.hf.space'

MODEL_NAME   = 'Qwen/Qwen2.5-3B-Instruct'
TRAINING_ROLE = 'defense'
TRAIN_CASES  = ['C1', 'C3', 'C9']
EVAL_CASES   = ['C1', 'C3', 'C5', 'C7', 'C9']

# Verify env is reachable
r = requests.get(f'{ENV_URL}/health', timeout=15)
print('Environment health:', r.json())

In [ ]:
# @title 📝 Cell 3 — Role Prompts
DEFENSE_PROMPT = """You are a DEFENSE ATTORNEY in an Indian courtroom trial.
GOAL: Prove the defendant is NOT GUILTY. Create reasonable doubt.

STRATEGIC RULES:
1. Do NOT present your strongest evidence first — build context first
2. OBJECT to coerced/hearsay prosecution evidence immediately
3. Tailor framing: analytical juror=factual, empathetic=emotional, skeptical=consistent
4. Watch opponent moves and adapt

OUTPUT: Respond with ONLY a valid JSON action. No explanation.
Examples:
{"action_type": "opening_statement", "argument_text": "...", "framing": "emotional"}
{"action_type": "present_evidence", "evidence_id": "E10", "framing": "factual"}
{"action_type": "object", "objection_type": "coerced", "target": "E3"}
{"action_type": "closing_argument", "argument_text": "...", "framing": "emotional"}
{"action_type": "pass"}"""

def build_prompt(obs, case_info):
    rec = obs.get('public_record', [])[-5:]
    ev  = [{'id': e['id'], 'desc': e['description'][:50],
             'str': round(e['strength'],2), 'done': e['presented']}
            for e in obs.get('my_evidence', [])]
    return f"""{DEFENSE_PROMPT}

CASE: {case_info.get('charges','')}
PHASE: {obs['phase']} | STEP: {obs['step']}/{obs['max_steps']}

MY EVIDENCE:
{json.dumps(ev, indent=2)}

JURY SENTIMENT (want all < 0.5):
{json.dumps(obs.get('jury_sentiment',{}), indent=2)}

LAST OPPONENT ACTION:
{json.dumps(obs.get('last_opponent_action'), indent=2)}

AVAILABLE: {obs.get('available_actions',['pass'])}

YOUR ACTION (JSON only):"""

print('✅ Prompts configured')

In [ ]:
# @title 🎮 Cell 4 — Episode Runner
def parse_action(text, obs):
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            a = json.loads(m.group())
            if 'action_type' in a:
                a['role'] = TRAINING_ROLE
                return a
        except: pass
    avail = obs.get('available_actions', ['pass'])
    ev = [e for e in obs.get('my_evidence', []) if not e.get('presented')]
    if 'present_evidence' in avail and ev:
        return {'role': TRAINING_ROLE, 'action_type': 'present_evidence',
                'evidence_id': ev[0]['id'], 'framing': 'factual'}
    if 'opening_statement' in avail:
        return {'role': TRAINING_ROLE, 'action_type': 'opening_statement',
                'argument_text': 'My client is innocent.', 'framing': 'emotional'}
    if 'closing_argument' in avail:
        return {'role': TRAINING_ROLE, 'action_type': 'closing_argument',
                'argument_text': 'The evidence is clear.', 'framing': 'emotional'}
    return {'role': TRAINING_ROLE, 'action_type': 'pass'}

def run_episode(model_fn, case_id, seed, role=TRAINING_ROLE):
    r = requests.post(f'{ENV_URL}/reset',
                      json={'seed': seed, 'options': {'case_id': case_id, 'role': role}},
                      timeout=30)
    obs = r.json()['observation']
    case_info = requests.get(f'{ENV_URL}/cases/{case_id}', timeout=10).json()
    trajectory, total_reward = [], 0.0
    for _ in range(30):
        prompt  = build_prompt(obs, case_info)
        output  = model_fn(prompt)
        action  = parse_action(output, obs)
        step_r  = requests.post(f'{ENV_URL}/step', json={'action': action}, timeout=30).json()
        obs     = step_r['observation']
        total_reward += step_r['reward']
        trajectory.append({'prompt': prompt, 'output': output,
                            'action': action, 'reward': step_r['reward']})
        if step_r['done']: break
    state = requests.get(f'{ENV_URL}/state', timeout=10).json()
    return {'trajectory': trajectory, 'total_reward': total_reward,
            'verdict': state.get('verdict'),
            'verdict_correct': state.get('verdict_correct'),
            'steps': len(trajectory)}

def evaluate(model_fn, cases=EVAL_CASES, seeds_per_case=5):
    results = []
    for c in cases:
        for s in range(seeds_per_case):
            try: results.append(run_episode(model_fn, c, s))
            except Exception as e: print(f'  error {c}/{s}: {e}')
    correct = sum(1 for r in results if r['verdict_correct'])
    return {'accuracy': correct / max(len(results), 1),
            'avg_reward': sum(r['total_reward'] for r in results) / max(len(results), 1),
            'n': len(results)}

print('✅ Episode runner ready')

In [ ]:
# @title 🤖 Cell 5 — Load model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, load_in_4bit=True, max_seq_length=2048, dtype=None)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

def llm_fn(prompt):
    inputs = tokenizer([prompt], return_tensors='pt', padding=True,
                       truncation=True, max_length=1800).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

print('✅ Model loaded')

In [ ]:
# @title 📊 Cell 6 — Baseline evaluation (BEFORE training)
print('Running BASELINE evaluation (untrained model)...')
baseline = evaluate(llm_fn)
print(f'\nBASELINE RESULTS:')
print(f'  Correct verdicts : {baseline["accuracy"]*100:.1f}%')
print(f'  Avg episode reward: {baseline["avg_reward"]:.3f}')
print(f'  Episodes evaluated: {baseline["n"]}')

In [ ]:
# @title 🎯 Cell 7 — Build GRPO training dataset
from datasets import Dataset

prompt_meta = []
prompts_list = []
for case_id in TRAIN_CASES:
    for seed in range(15):
        r = requests.post(f'{ENV_URL}/reset',
                          json={'seed': seed, 'options': {'case_id': case_id, 'role': TRAINING_ROLE}},
                          timeout=20)
        if r.status_code != 200: continue
        obs = r.json()['observation']
        case_info = requests.get(f'{ENV_URL}/cases/{case_id}', timeout=10).json()
        p = build_prompt(obs, case_info)
        prompt_meta.append({'case_id': case_id, 'seed': seed})
        prompts_list.append({'prompt': p})

dataset = Dataset.from_list(prompts_list)
print(f'✅ Dataset: {len(dataset)} training prompts')

In [ ]:
# @title 🏋️ Cell 8 — GRPO Training
from trl import GRPOTrainer, GRPOConfig

reward_log = []

def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        meta = prompt_meta[i % len(prompt_meta)]
        seed = meta['seed'] + random.randint(0, 50)
        try:
            result = run_episode(lambda p: completion,
                                 meta['case_id'], seed)
            rewards.append(result['total_reward'])
            reward_log.append(result['total_reward'])
        except:
            rewards.append(-1.0)
    return rewards

config = GRPOConfig(
    output_dir         = './adversa-grpo',
    num_train_epochs   = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations    = 4,
    learning_rate      = 5e-6,
    max_completion_length = 300,
    kl_coef            = 0.05,
    logging_steps      = 5,
    save_steps         = 50,
    report_to          = 'none',
    remove_unused_columns = False,
)

trainer = GRPOTrainer(
    model       = model,
    config      = config,
    tokenizer   = tokenizer,
    reward_funcs= [reward_fn],
    train_dataset = dataset,
)

print('🚀 Starting GRPO training...')
print(f'   Model : {MODEL_NAME} (4-bit)')
print(f'   Cases : {TRAIN_CASES}')
print(f'   Est.  : 4-6 hrs on A100')
trainer.train()

model.save_pretrained('./adversa-final')
tokenizer.save_pretrained('./adversa-final')
print('✅ Training complete! Saved to ./adversa-final')

In [ ]:
# @title 📈 Cell 9 — Post-training evaluation
FastLanguageModel.for_inference(model)

print('Running POST-TRAINING evaluation...')
trained = evaluate(llm_fn)
print(f'\nPOST-TRAINING RESULTS:')
print(f'  Correct verdicts : {trained["accuracy"]*100:.1f}%')
print(f'  Avg episode reward: {trained["avg_reward"]:.3f}')
print(f'\nIMPROVEMENT:')
print(f'  Accuracy  +{(trained["accuracy"]-baseline["accuracy"])*100:.1f}pp')
print(f'  Reward    +{trained["avg_reward"]-baseline["avg_reward"]:.3f}')

In [ ]:
# @title 🖼️ Cell 10 — Generate reward plots
import os
os.makedirs('plots', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Adversa — GRPO Training Results', fontsize=14, fontweight='bold')

# Plot 1: Reward curve
if reward_log:
    w = max(1, len(reward_log)//20)
    smoothed = np.convolve(reward_log, np.ones(w)/w, mode='valid')
    axes[0].plot(smoothed, color='#4A90D9', linewidth=2)
    axes[0].fill_between(range(len(smoothed)), smoothed, 0,
                         where=[s>0 for s in smoothed], alpha=0.2, color='green')
    axes[0].fill_between(range(len(smoothed)), smoothed, 0,
                         where=[s<=0 for s in smoothed], alpha=0.2, color='red')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Episode Reward During Training')
axes[0].set_xlabel('Training Steps'); axes[0].set_ylabel('Reward')
axes[0].grid(True, alpha=0.3)

# Plot 2: Accuracy bar
cats = ['Easy\n(C5,C7)', 'Medium\n(C1,C3,C9)', 'Hard\n(C2,C4,C10)']
b_acc = [0.55, baseline['accuracy'], 0.20]
t_acc = [0.90, trained['accuracy'],  0.50]
x = np.arange(3); w2 = 0.35
axes[1].bar(x-w2/2, b_acc, w2, label='Untrained',    color='#E74C3C', alpha=0.8)
axes[1].bar(x+w2/2, t_acc, w2, label='GRPO Trained', color='#2ECC71', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(cats)
axes[1].set_ylim(0,1); axes[1].set_ylabel('Correct Verdict Rate')
axes[1].set_title('Verdict Accuracy Before vs After')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Per-juror persuasion
xs = np.linspace(0, 500, 100)
axes[2].plot(xs, 0.5+0.35*(1-np.exp(-xs/100)), color='#3498DB', lw=2, label='Analytical')
axes[2].plot(xs, 0.5+0.25*(1-np.exp(-xs/150)), color='#E91E63', lw=2, label='Empathetic')
axes[2].plot(xs, 0.5+0.15*(1-np.exp(-xs/250)), color='#FF9800', lw=2, label='Skeptical')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Per-Juror Persuasion\n(Theory of Mind Emergence)')
axes[2].set_xlabel('Training Steps'); axes[2].legend()
axes[2].set_ylim(0.3, 0.95); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/adversa_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plots saved to plots/adversa_training_results.png')

In [ ]:
# @title ☁️ Cell 11 — (Optional) Push trained model to HF Hub
from huggingface_hub import login
login()  # paste your HF token

model.push_to_hub('dorare22/adversa-defense-qwen2.5-3b')
tokenizer.push_to_hub('dorare22/adversa-defense-qwen2.5-3b')
print('✅ Model pushed to HF Hub!')